# Meme Hunter MFE 

**Architecture:**
- Target Labeling: Maximum Forward Excursion (MFE) / Horizon 12h / Min Pump 10%
- LightGBM Classifier (Binary)
- Walk-Forward Expanding Window Filter + 15 Day Purge Gap
- Hit & Run 10% Features: Micro-Volume, Price Acceleration, Order Flow Proxy + Market Context
- Output: Probability of +10% pump over next 12h

In [ ]:
import os, sys, glob
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

TIMEFRAME = '1h'
IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

WORKING_DIR = Path('/kaggle/working') if IS_KAGGLE else Path('./ml/training/models')
WORKING_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = Path('/kaggle/input/datasets/hungbui317/macd-coin/data') if IS_KAGGLE else Path('./data')
OHLCV_DIR = DATA_DIR / 'ohlcv'
OUTPUT_FILE = str(WORKING_DIR / f'features_{TIMEFRAME}_full.parquet')

TF_CONFIG = {
    '1h':  {'rule': '1h',  'min_bars': 300, 'atr_clamp': (0.005, 0.15), 'max_tp': 1.00, 'max_bars': 48, 'unit': '1h bars'},
    '4h':  {'rule': '4h',  'min_bars': 200, 'atr_clamp': (0.002, 0.06), 'max_tp': 0.15, 'max_bars': 30, 'unit': '4h bars'},
    '8h':  {'rule': '8h',  'min_bars': 100, 'atr_clamp': (0.003, 0.08), 'max_tp': 0.20, 'max_bars': 20, 'unit': '8h bars'},
    '12h': {'rule': '12h', 'min_bars': 80,  'atr_clamp': (0.004, 0.10), 'max_tp': 0.25, 'max_bars': 15, 'unit': '12h bars'},
    '1d':  {'rule': '1D',  'min_bars': 100, 'atr_clamp': (0.005, 0.13), 'max_tp': 0.30, 'max_bars': 15, 'unit': 'days'},
}
CFG = TF_CONFIG[TIMEFRAME]
print(f"Timeframe: {TIMEFRAME} | Output: {OUTPUT_FILE}")
if OHLCV_DIR.exists(): print(f"Found {len(list(OHLCV_DIR.glob('*.parquet')))} symbol files")
else: print(f"⚠️ {OHLCV_DIR} not found!")

In [ ]:
def load_ohlcv_1h(symbol):
    for name in [f"{symbol}_USDT.parquet", f"{symbol}.parquet"]:
        fp = OHLCV_DIR / name
        if fp.exists():
            df = pd.read_parquet(fp)
            if 'timestamp' not in df.columns and 'open_time' in df.columns: df = df.rename(columns={'open_time':'timestamp'})
            df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms') if df['timestamp'].dtype=='int64' else pd.to_datetime(df['timestamp'])
            return df.sort_values('timestamp').reset_index(drop=True)
    return pd.DataFrame()
def resample_1h(df_1h, tf):
    if df_1h.empty: return pd.DataFrame()
    return df_1h.set_index('timestamp').resample(TF_CONFIG[tf]['rule']).agg({'open':'first','high':'max','low':'min','close':'last','volume':'sum'}).dropna().reset_index()
def calculate_rsi(prices, period=14):
    d = prices.diff(); g = d.where(d>0,0).rolling(period).mean(); l = (-d.where(d<0,0)).rolling(period).mean()
    return 100-(100/(1+g/(l.replace(0,np.nan)+1e-9)))
def calculate_macd(df, fast=12, slow=26, signal=9):
    ef=df['close'].ewm(span=fast).mean(); es=df['close'].ewm(span=slow).mean()
    df['macd']=ef-es; df['macd_signal']=df['macd'].ewm(span=signal).mean(); df['macd_histogram']=df['macd']-df['macd_signal']
    df['macd_cross_up']=((df['macd']>df['macd_signal'])&(df['macd'].shift(1)<=df['macd_signal'].shift(1))).astype(int)
    df['macd_cross_down']=((df['macd']<df['macd_signal'])&(df['macd'].shift(1)>=df['macd_signal'].shift(1))).astype(int)
    df['macd_slope']=df['macd'].diff(); df['macd_acceleration']=df['macd_slope'].diff(); return df
def calculate_liquidity_sweep(df, lookback=20):
    df = df.copy()
    df['swing_low'] = df['low'].rolling(window=lookback).min().shift(1)
    df['swing_high'] = df['high'].rolling(window=lookback).max().shift(1)
    df['candle_range'] = df['high'] - df['low'] + 1e-9
    df['lower_wick'] = df[['open', 'close']].min(axis=1) - df['low']
    df['upper_wick'] = df['high'] - df[['open', 'close']].max(axis=1)
    df['lower_wick_ratio'] = df['lower_wick'] / df['candle_range']
    df['upper_wick_ratio'] = df['upper_wick'] / df['candle_range']
    df['vol_sma_20'] = df['volume'].rolling(20).mean()
    cond_sweep_bottom = df['low'] < df['swing_low']
    cond_reject_bottom = df['close'] > df['swing_low']
    cond_pinbar_bottom = df['lower_wick_ratio'] > 0.3
    cond_vol_surge = df['volume'] > df['vol_sma_20']
    df['bullish_sweep'] = (cond_sweep_bottom & cond_reject_bottom & cond_pinbar_bottom & cond_vol_surge).astype(int)
    cond_sweep_top = df['high'] > df['swing_high']
    cond_reject_top = df['close'] < df['swing_high']
    cond_pinbar_top = df['upper_wick_ratio'] > 0.3
    df['bearish_sweep'] = (cond_sweep_top & cond_reject_top & cond_pinbar_top & cond_vol_surge).astype(int)
    df['macd_cross_up'] = df['bullish_sweep']
    df['macd_cross_down'] = df['bearish_sweep']
    return df
def calculate_features(df):
    df=df.copy(); df['log_returns']=np.log(df['close']/df['close'].shift(1))
    df['high_low_range']=(df['high']-df['low'])/df['close']; df['body_size']=abs(df['close']-df['open'])/df['close']
    df['candle_range']=df['high']-df['low']+1e-9; df['lower_wick']=df[['open','close']].min(axis=1)-df['low']; df['upper_wick']=df['high']-df[['open','close']].max(axis=1)
    df['lower_wick_ratio_current']=df['lower_wick']/df['candle_range']; df['upper_wick_ratio_current']=df['upper_wick']/df['candle_range']
    for p in [7,14,21,50,100,200]: df[f'ema_{p}']=df['close'].ewm(span=p).mean()
    for p in [10,20,50,200]: df[f'sma_{p}']=df['close'].rolling(p).mean()
    tr=pd.concat([df['high']-df['low'],abs(df['high']-df['close'].shift(1)),abs(df['low']-df['close'].shift(1))],axis=1).max(axis=1)
    df['atr_14']=tr.rolling(14).mean(); df['volatility_14']=df['log_returns'].rolling(14).std()
    df['vol_sma_14']=df['volatility_14'].rolling(14).mean(); df['vol_compression']=df['volatility_14']/(df['vol_sma_14']+1e-9)
    df['volume_sma_20']=df['volume'].rolling(20).mean(); df['volume_std_20']=df['volume'].rolling(20).std()
    df['volume_ratio']=df['volume']/(df['volume_sma_20']+1e-9); df['volume_zscore']=(df['volume']-df['volume_sma_20'])/(df['volume_std_20']+1e-9)
    df['volume_trend']=df['volume'].rolling(7).mean()/(df['volume'].rolling(21).mean()+1e-9); df['volume_spike']=(df['volume_ratio']>2).astype(int)
    df['rsi_14']=calculate_rsi(df['close'], 14); df['rsi_slope']=df['rsi_14'].diff(3)
    l14=df['low'].rolling(14).min(); h14=df['high'].rolling(14).max()
    df['stoch_k']=100*(df['close']-l14)/(h14-l14).replace(0,np.nan); df['stoch_d']=df['stoch_k'].rolling(3).mean()
    df['roc_7']=df['close'].pct_change(7); df['roc_14']=df['close'].pct_change(14)
    # Phase 11 Features
    df['sma_30']=df['close'].rolling(30).mean(); df['price_vs_sma_30']=df['close']/(df['sma_30']+1e-9)
    df['momentum_30']=df['close'].pct_change(30)
    pdm=df['high'].diff(); mdm=-df['low'].diff()
    pdm=pdm.where((pdm>mdm)&(pdm>0),0); mdm=mdm.where((mdm>pdm)&(mdm>0),0); atr_s=tr.rolling(14).mean()
    pdi=100*(pdm.rolling(14).mean()/atr_s.replace(0,np.nan)); mdi=100*(mdm.rolling(14).mean()/atr_s.replace(0,np.nan))
    df['adx']=(100*abs(pdi-mdi)/(pdi+mdi).replace(0,np.nan)).rolling(14).mean()
    df['dist_to_high_30d']=(df['close']-df['high'].rolling(30).max())/df['close']
    df['dist_to_low_30d']=(df['close']-df['low'].rolling(30).min())/df['close']
    for e in [21,50,200]: df[f'dist_to_ema_{e}_pct']=(df['close']-df[f'ema_{e}'])/df['close']
    df['trend_state']=np.where(df['close']>df['sma_50'],1,np.where(df['close']<df['sma_50'],-1,0))
    df['is_trending']=(df['adx']>25).astype(int); df['is_volatile']=(df['vol_compression']>1.5).astype(int)
    df['hour_sin']=np.sin(2*np.pi*df['timestamp'].dt.hour/24); df['hour_cos']=np.cos(2*np.pi*df['timestamp'].dt.hour/24)
    df['day_sin']=np.sin(2*np.pi*df['timestamp'].dt.dayofweek/7); df['day_cos']=np.cos(2*np.pi*df['timestamp'].dt.dayofweek/7)
    df['vol_ratio_alpha']=df['volume_ratio']*df['volatility_14']
    df['market_structure_bull']=((df['close']>df['sma_200'])&(df['sma_50']>df['sma_200'])).astype(int)
    bb_mid=df['close'].rolling(20).mean(); bb_std=df['close'].rolling(20).std()
    bb_wd=(bb_mid+2*bb_std - (bb_mid-2*bb_std))/(bb_mid+1e-9)
    df['bb_squeeze']=(bb_wd<bb_wd.rolling(20).quantile(0.2)).astype(int)
    df['vwap_30d']=(df['close']*df['volume']).rolling(30).sum()/(df['volume'].rolling(30).sum()+1e-9)
    df['above_poc']=(df['close']>df['vwap_30d']).astype(int)
    
    # Hit & Run 10% Features
    df['micro_volume']=df['volume']/(df['volume'].rolling(5).mean()+1e-9)
    df['price_accel']=df['close'].pct_change(1)/(df['close'].pct_change(4).replace(0,np.nan)+1e-9)
    df['order_flow_proxy']=(df['close']-df['low'])/(df['high']-df['low']+1e-9)
    
    df=calculate_macd(df); df=df.drop(columns=['macd_cross_up','macd_cross_down'], errors='ignore')
    df=calculate_liquidity_sweep(df)
    df['usd_vol_24h'] = (df['volume'] * df['close']).rolling(24).sum()
    return df.dropna(subset=['macd','swing_low','vol_sma_20','vwap_30d','usd_vol_24h'])
def generate_momentum_labels(df, horizon=12, min_pump=0.10):
    df = df.copy()
    
    # 1. Đảo ngược Data để nhìn về tương lai
    df_rev = df.iloc[::-1].copy()
    
    # 2. Tìm Đỉnh cao nhất (Max High) trong 'horizon' nến tiếp theo
    if 'symbol' in df_rev.columns:
        future_max_high = df_rev.groupby('symbol', group_keys=False)['high'].apply(lambda x: x.rolling(horizon, min_periods=1).max())
    else:
        future_max_high = df_rev['high'].rolling(horizon, min_periods=1).max()
        
    df['future_max_high'] = future_max_high.sort_index()
    df['max_pump_pct'] = (df['future_max_high'] - df['close']) / df['close']
    df['label'] = (df['max_pump_pct'] >= min_pump).astype(int)
    
    if 'usd_vol_24h' in df.columns:
        df.loc[df['usd_vol_24h'] < 1000000, 'label'] = np.nan
        
    df['ignition'] = df['label']
    if 'symbol' in df.columns:
        df['future_return'] = df.groupby('symbol')['close'].shift(-horizon) / df['close'] - 1
    else:
        df['future_return'] = df['close'].shift(-horizon) / df['close'] - 1
    df['trade_result'] = np.where(df['label'] == 1, 'WIN', 'LOSS')
    return df.drop(columns=['future_max_high'])
def apply_winsorization(df,fc,lo=0.01,hi=0.99):
    df=df.copy()
    for c in fc:
        if c in df.columns and df[c].dtype in ['float64','float32','int64']: l,h=df[c].quantile(lo),df[c].quantile(hi); df[c]=df[c].clip(l,h)
    return df
def apply_feature_shift(df):
    ex={'timestamp','symbol','open','high','low','close','volume','label','ignition','trade_result','macd_cross_up','macd_cross_down'}
    sc=[c for c in df.columns if c not in ex]
    if 'symbol' in df.columns: df[sc]=df.groupby('symbol')[sc].shift(1)
    else: df[sc]=df[sc].shift(1)
    return df.dropna(subset=sc[:3])
    
def reduce_mem_usage(df):
    """Giảm dung lượng RAM bằng cách ép kiểu dữ liệu."""
    for col in df.columns:
        col_type = df[col].dtype
        
        if not pd.api.types.is_numeric_dtype(col_type):
            continue
            
        try:
            c_min = df[col].min()
            c_max = df[col].max()
            if pd.api.types.is_integer_dtype(col_type):
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max: df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max: df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max: df[col] = df[col].astype(np.int32)
                else: df[col] = df[col].astype(np.int64)
            else:
                df[col] = df[col].astype(np.float32) # Luôn cast float về float32 để tránh lỗi LGBM
        except Exception as e:
            pass
    return df

print("✓ Pipeline loaded.")

def scan_historical_confluence(df, horizon=12):
    """
    Quét toàn bộ lịch sử để tìm các điểm Hợp Lưu Breakout kinh điển.
    """
    df = df.copy()
    if 'rsi_14' not in df.columns:
        from ml.data_pipeline import calculate_rsi
        df['rsi_14'] = calculate_rsi(df['close'], 14)
        
    df['resistance_50'] = df.groupby('symbol')['high'].transform(lambda x: x.rolling(50).max().shift(1))
    df['ema_20'] = df.groupby('symbol')['close'].transform(lambda x: x.ewm(span=20).mean())
    df['ema_50'] = df.groupby('symbol')['close'].transform(lambda x: x.ewm(span=50).mean())
    
    cond_uptrend = (df['ema_20'] > df['ema_50']) & (df['low'] > df['ema_50'])
    cond_rsi = df['rsi_14'] > 60
    vol_sma_20 = df.groupby('symbol')['volume'].transform(lambda x: x.rolling(20).mean().shift(1))
    cond_volume = df['volume'] > (vol_sma_20 * 2.5)
    cond_breakout = df['close'] > df['resistance_50']
    
    df['is_golden_setup'] = cond_uptrend & cond_rsi & cond_volume & cond_breakout
    
    # Tính thực tế bay bao nhiêu %
    df['actual_pump_pct'] = df.groupby('symbol')['high'].transform(lambda x: (x.shift(-horizon).rolling(horizon).max() - df['close']) / df['close'])
    
    signals = df[df['is_golden_setup'] == True].copy()
    cols = ['timestamp', 'symbol', 'close', 'resistance_50', 'volume', 'actual_pump_pct']
    return signals[cols]

In [ ]:
symbols=[f.stem.replace('_USDT','') for f in OHLCV_DIR.glob('*.parquet')]
symbols=[s for s in symbols if not any(x in s for x in ['-26','-25','-24'])]
print(f"Found {len(symbols)} symbols")
btc_context=pd.DataFrame()
btc_sym='BTCUSDT' if 'BTCUSDT' in symbols else ('BTC' if 'BTC' in symbols else None)
if btc_sym:
    btc_df=calculate_features(resample_1h(load_ohlcv_1h(btc_sym),TIMEFRAME))
    btc_context=btc_df[['timestamp','close','sma_200','adx','log_returns']].copy()
    btc_context.columns=['timestamp','btc_close','btc_sma_200','btc_adx','btc_returns']
    btc_context['btc_is_bull_regime']=(btc_context['btc_close']>btc_context['btc_sma_200']).astype(int)
    btc_context['btc_trend_strength']=np.where(btc_context['btc_adx']>25,1,0)
all_data=[]
for sym in symbols:
    try:
        d1=load_ohlcv_1h(sym)
        if d1.empty: continue
        dt=resample_1h(d1,TIMEFRAME)
        if len(dt)<CFG['min_bars']: continue
        dt['symbol']=sym; dt=calculate_features(dt); dt['is_bullish_cross']=dt['macd_cross_up'].values
        # Phase 11: Add Multi-Timeframe (1D) Context
        d1d=resample_1h(d1,'1d')
        d1d['ema_200_1d']=d1d['close'].ewm(span=200).mean()
        d1d['rsi_14_1d']=calculate_rsi(d1d['close'],14)
        d1d['ema_200_1d_dist']=(d1d['close']-d1d['ema_200_1d'])/d1d['close']
        d1d_feat=d1d[['timestamp','ema_200_1d_dist','rsi_14_1d']].copy()
        d1d_feat['date']=d1d_feat['timestamp'].dt.date
        d1d_feat=d1d_feat.drop(columns='timestamp').shift(1) # Prevent lookahead
        dt['date']=dt['timestamp'].dt.date
        dt=dt.merge(d1d_feat,on='date',how='left').drop(columns='date')
        for c in ['ema_200_1d_dist','rsi_14_1d']: dt[c]=dt[c].ffill().fillna(0.5 if 'rsi' in c else 0)
        
        if not btc_context.empty:
            dt=dt.merge(btc_context,on='timestamp',how='left')
            for c in ['btc_is_bull_regime','btc_trend_strength','btc_returns']: dt[c]=dt[c].ffill().fillna(0)
            dt['rs_vs_btc']=dt['log_returns']-dt['btc_returns']; dt['rs_vs_btc_sma7']=dt['rs_vs_btc'].rolling(7).mean()
            dt['btc_corr']=dt['log_returns'].rolling(14).corr(dt['btc_returns']).fillna(0)
            
        # Đẩy các bước xử lý (shift, memory, label) vào mức độ single-symbol
        dt = apply_feature_shift(dt)
        if dt.empty: continue
        
        dt = reduce_mem_usage(dt) # Tối ưu RAM ngay tại đây
        dt = generate_momentum_labels(dt, horizon=12, min_pump=0.10)
        
        all_data.append(dt)
        print(f"  ✓ {sym}: {len(dt)} {CFG['unit']}")
        
        # Batching: Nối và giải phóng RAM mỗi 50 symbols
        if len(all_data) >= 50 or sym == symbols[-1]:
            import gc
            df_batch = pd.concat(all_data, ignore_index=True)
            df_batch.to_parquet(WORKING_DIR / f'batch_features_{sym}.parquet', index=False)
            all_data = [] # Giải phóng mảng lớn
            del df_batch
            gc.collect()
            
    except Exception as e: print(f"  ✗ {sym}: {e}")
    
# Thu gom tất cả các file parquet
import glob
print("\n✓ Merging all batches from disk...")
batch_files = glob.glob(str(WORKING_DIR / 'batch_features_*.parquet'))
df = pd.read_parquet(batch_files)

# Dọn dẹp file tạm
for f in batch_files: os.remove(f)

print(f"✓ Combined: {len(df)} rows")
df.to_parquet(OUTPUT_FILE,index=False); print(f"\n✅ Saved to {OUTPUT_FILE} ({len(df)} rows)")
y=df['label'].dropna().values
print(f"Labels Return: Min={y.min():.3f}, Max={y.max():.3f}, Mean={y.mean():.3f}, Median={np.median(y):.3f}")


from sklearn.preprocessing import StandardScaler, LabelEncoder, RobustScaler
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
import joblib

In [ ]:
MODEL_FEATURES=[
    'rsi_14','rsi_slope','stoch_k','stoch_d','roc_7','roc_14',
    'volume_ratio','volume_zscore','volume_trend','rs_vs_btc','rs_vs_btc_sma7','vol_compression',
    'dist_to_high_30d','dist_to_low_30d','dist_to_ema_21_pct','dist_to_ema_50_pct','dist_to_ema_200_pct',
    'price_vs_sma_30','momentum_30','macd_slope','macd_acceleration',
    'lower_wick_ratio_current','upper_wick_ratio_current',
    'bb_squeeze','above_poc',
    'micro_volume','price_accel','order_flow_proxy',
    'btc_is_bull_regime','btc_trend_strength','adx','hour_sin','hour_cos','day_sin','day_cos',
    'btc_corr','trend_state','is_trending','is_volatile','ema_200_1d_dist','rsi_14_1d'
]

def prepare_data(tf):
    paths=[f'/kaggle/working/features_{tf}_full.parquet',f'./data/processed/features_{tf}_full.parquet', OUTPUT_FILE]
    path=next((p for p in paths if os.path.exists(p)),None)
    if not path: raise FileNotFoundError(f"No data! Run Pipeline. Looked: {paths}")
    
    df = pd.read_parquet(path).sort_values(['symbol', 'timestamp']).reset_index(drop=True)
    mask = df['label'].notnull() & df['usd_vol_24h'].notnull()
    
    idx_all = df[mask].index.tolist()
    print(f"Dataset prepared! Total valid valid samples: {len(idx_all)}")
    return df, idx_all


def train_and_evaluate_window(df, indices_tr, indices_te):
    # Dùng list features đã lọc
    features = [f for f in MODEL_FEATURES if f in df.columns]
    
    X_tr = df.loc[indices_tr, features].fillna(0)
    y_tr = df.loc[indices_tr, 'label'].values
    X_te = df.loc[indices_te, features].fillna(0)
    y_te = df.loc[indices_te, 'label'].values
    
    lgbm_model = lgb.LGBMClassifier(
        n_estimators=500, learning_rate=0.03, max_depth=5, 
        objective='binary', random_state=42, n_jobs=-1, verbose=-1,
        class_weight='balanced'
    )
    
    lgbm_model.fit(X_tr, y_tr)
    preds_te = lgbm_model.predict_proba(X_te)[:, 1]
    
    def evaluate_auc(y_true, y_p):
        if len(np.unique(y_true)) < 2: return 0.0
        return roc_auc_score(y_true, y_p)
        
    ic = evaluate_auc(y_te, preds_te)
    
    # Tính Precision ở Top 1% những kèo mô hình tự tin nhất
    threshold = np.percentile(preds_te, 99) 
    bot_calls = (preds_te >= threshold) 
    true_pumps = y_te[bot_calls]
    precision_top1 = np.mean(true_pumps) if len(true_pumps) > 0 else 0.0
    
    print(f"   >>> OOS ROC AUC: {ic:.4f} | Precision@Top1%: {precision_top1:.4f}")
    
    df_oos = pd.DataFrame({
        'timestamp': df.loc[indices_te, 'timestamp'].values,
        'symbol': df.loc[indices_te, 'symbol'].values,
        'pred_ens': preds_te,
        'label': y_te
    })
    
    return ic, precision_top1, lgbm_model, df_oos, features

def train_momentum_lgbm(tf='1h'):
    print(f"\n{'='*60}\nKHỞI ĐỘNG HỆ THỐNG MFE QUANT ({tf})\n{'='*60}")
    
    df, idx_all = prepare_data(tf)
    
    # Filter data to post-2023 to avoid "Garbage Memory" of 2020-2022
    timestamps = df.loc[idx_all, 'timestamp']
    valid_mask = timestamps >= pd.Timestamp('2023-01-01')
    idx_all = [i for i, m in zip(idx_all, valid_mask) if m]
    timestamps = df.loc[idx_all, 'timestamp']
    
    start_date = timestamps.min()
    end_date = timestamps.max()
    
    print(f"\n[PHASE A] Chạy Walk-Forward Validation ({start_date.date()} -> {end_date.date()})")
    
    train_months = 6
    test_months = 2
    horizon_gap = pd.DateOffset(days=15) # PURGE GAP BẮT BUỘC: 15 ngày
    
    train_end = start_date + pd.DateOffset(months=train_months) 
    wf_auc = []
    wf_precision = []
    all_oos_dfs = []
    
    while True:
        test_end = train_end + pd.DateOffset(months=test_months)
        if test_end > end_date: break
            
        i_tr = timestamps[(timestamps >= start_date) & (timestamps < (train_end - horizon_gap))].index.tolist()
        i_te = timestamps[(timestamps >= train_end) & (timestamps < test_end)].index.tolist()
        
        if len(i_tr) > 200 and len(i_te) > 20:
            print(f"📍 Cửa sổ: Train ({start_date.date()}->{(train_end - horizon_gap).date()}) | Test ({train_end.date()}->{test_end.date()})")
            ic, precision, df_oos, _ = train_and_evaluate_window(df, i_tr, i_te)
            wf_auc.append(ic)
            wf_precision.append(precision)
            all_oos_dfs.append(df_oos)
            
        train_end = test_end
        
    print(f"\n{'='*60}\n[PHASE B] TRAIN FINAL MODEL (Production)\n{'='*60}")
    
    # Train final data từ start_date -> end_date - horizon_gap
    final_train_end = end_date - horizon_gap
    i_final = timestamps[timestamps < final_train_end].index.tolist()
    
    print(f"Huấn luyện mô hình Production trên {len(i_final)} mẫu (tất cả data trừ {horizon_gap.days} ngày cuối).")
    _, lgbm_final, _, features_final = train_and_evaluate_window(df, i_final, i_final)
    
    print(f"\n✅ ĐÃ HOÀN TẤT HUẤN LUYỆN!")
    if wf_auc:
        print(f"   => Average WFV ROC AUC: {np.mean(wf_auc):.4f}")
        print(f"   => Average WFV Precision@Top1%: {np.mean(wf_precision):.4f}")
    
    # Run Confluence Scan on the OOS data for sanity check
    if all_oos_dfs:
        print(f"\n[PHASE C] Historical Confluence Scan (Manual Verification)")
        full_oos = pd.concat(all_oos_dfs)
        # We need OHLC data too for Confluence, but OOS only has pred/label. 
        # So we scan the full df for confluence symbols found in signals.
        signals = scan_historical_confluence(df[df['timestamp'] >= start_date])
        print(f"Tìm thấy {len(signals)} kèo Hợp lưu (Golden Setup) trong lịch sử!")
        if not signals.empty:
            print(signals.sort_values('actual_pump_pct', ascending=False).head(10))
    
    import shutil
    tf_dir = Path(f"{WORKING_DIR}/models/{tf}")
    tf_dir.mkdir(parents=True, exist_ok=True)
    
    if all_oos_dfs:
        final_oos_df = pd.concat(all_oos_dfs, ignore_index=True)
        final_oos_df = final_oos_df.sort_values(['symbol', 'timestamp']).reset_index(drop=True)
        final_oos_df.to_csv(tf_dir / 'oos_predictions.csv', index=False)
        print(f"\n✅ Đã lưu {len(final_oos_df)} dòng dự báo OOS vào oos_predictions.csv")
    
    joblib.dump(lgbm_final, tf_dir / 'ensemble_lgbm_tabular.joblib')
    joblib.dump(features_final, tf_dir / 'ensemble_meta.joblib')
    
    zip_filename = f"{WORKING_DIR}/ranker_production_{tf}"
    shutil.make_archive(zip_filename, 'zip', tf_dir)
    print(f"\n🚀 XONG! Sẵn sàng tải về: {zip_filename}.zip")
    
    if IS_KAGGLE:
        from IPython.display import display, FileLink
        display(FileLink(f"ranker_production_{tf}.zip"))

train_momentum_lgbm(TIMEFRAME)